In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy import stats
from shapely import wkt

# ======================
# Paths
# ======================
PROJECT_ROOT = Path(os.environ.get("PERSONALITY_SVI_ROOT", Path(__file__).resolve().parents[2]))
# V2: post-2010 personality data only (matches the paper's primary analysis)
IN_PARQUET = PROJECT_ROOT / "data/processed/model/final_dataset_zipcode_spatial2_env8_medianincome_racialdiversity_post2010.parquet"
OUT_DIR = PROJECT_ROOT / "reports/figures/city_maps_individual"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ======================
# Cities
# ======================
CITIES = [
    "austin_texas",
    "dallas_texas",
    "houston_texas",
    "san_antonio_texas",
]

# ======================
# Columns to plot
# ======================
ENV_COLS = [
    "env_greenery",
    "env_open_space",
    "env_building",
    "env_road",
    "env_active_infra",
    "env_active_presence",
    "env_vehicle_presence",
    "env_physical_boundaries",
    "env_symbolic_us_flag",
    "env_surveillance_cctv",
]

CENSUS_CONTROLS = [
    "visual_complexity_mean",
    "us_census_male_share",
    "us_census_age_15_29_share",
    "us_census_age_30_44_share",
    "us_census_age_45_59_share",
    "us_census_age_60_plus_share",
    "us_census_population_density_km2",
]

# Include raw median income for QA + existing z cols
ACS_COLS = [
    "acs_median_income",        # QA raw value (we will clean this)
    "acs_median_income_z",
    "acs_racial_diversity_z",
]

PLOT_VARS = ENV_COLS + CENSUS_CONTROLS + ACS_COLS

# Pretty labels (optional)
LABELS = {c: c for c in PLOT_VARS}
LABELS.update({
    "visual_complexity_mean": "Visual Complexity",
    "us_census_population_density_km2": "Population Density (per km²)",
    "acs_median_income": "ACS Median Income (raw, cleaned)",
    "acs_median_income_z": "Median Income (z, from parquet)",
    "acs_racial_diversity_z": "Racial Diversity (z, from parquet)",
})

# ======================
# Plot config
# ======================
Z_VMIN, Z_VMAX = -2, 2
CMAP = plt.cm.RdBu_r

# ======================
# Load parquet + geometry fix
# ======================
df = pd.read_parquet(IN_PARQUET)

if "geometry" not in df.columns:
    raise ValueError("Parquet missing 'geometry' column.")
if "city" not in df.columns:
    raise ValueError("Parquet missing 'city' column.")

# If geometry is WKT strings like "POLYGON ((...))"
sample = df["geometry"].dropna().iloc[0]
if isinstance(sample, str):
    df["geometry"] = df["geometry"].apply(lambda x: wkt.loads(x) if isinstance(x, str) else x)

gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

# Filter cities
gdf = gdf[gdf["city"].isin(CITIES)].copy()
if gdf.empty:
    print("Available city values (sample):", sorted(df["city"].unique().tolist())[:50])
    raise ValueError("After filtering, gdf is empty. Update CITIES to match parquet 'city' values.")

# Reproject for consistent display
gdf = gdf.to_crs(epsg=3857)

# ======================
# Precompute per-city extents so every figure uses identical framing
# ======================
city_info = {}
for city in CITIES:
    cg = gdf[gdf["city"] == city]
    bounds = cg.total_bounds  # xmin, ymin, xmax, ymax
    padding = 2000  # meters
    cx = (bounds[0] + bounds[2]) / 2
    cy = (bounds[1] + bounds[3]) / 2
    radius = max(bounds[2] - bounds[0], bounds[3] - bounds[1]) / 2 + padding
    city_info[city] = dict(center=(cx, cy), radius=radius)

# ======================
# Step 1: Clean obvious bad ACS median income values
#   - convert to numeric
#   - set <=0 to NaN (removes your extreme negative sentinel too)
# ======================
BAD_INCOME = "acs_median_income"
if BAD_INCOME in gdf.columns:
    gdf[BAD_INCOME] = pd.to_numeric(gdf[BAD_INCOME], errors="coerce")

    n_bad = (gdf[BAD_INCOME] <= 0).sum()
    print(f"[clean] {BAD_INCOME} <= 0 rows: {n_bad} (set to NaN)")
    gdf.loc[gdf[BAD_INCOME] <= 0, BAD_INCOME] = np.nan
else:
    print(f"[skip] {BAD_INCOME} not found in gdf")

# ======================
# Step 2: Robust global z-score helper
# ======================
def add_global_zscore(gdf_in: gpd.GeoDataFrame, col: str) -> str:
    """
    Compute global z-score for a column across ALL rows (all 4 cities together).
    NaNs are ignored; NaNs remain NaN in output.
    """
    zcol = f"{col}__z"

    if col not in gdf_in.columns:
        raise ValueError(f"Column not found: {col}")

    vals = pd.to_numeric(gdf_in[col], errors="coerce").to_numpy(dtype=float)
    z = stats.zscore(vals, nan_policy="omit")
    gdf_in[zcol] = z
    return zcol

# ======================
# Plot 1 variable per figure (4 cities in columns)
# ======================
def plot_variable_grid(gdf_in: gpd.GeoDataFrame, var: str, out_dir: Path):
    if var not in gdf_in.columns and var != "acs_median_income_z":
        print(f"[SKIP] missing column: {var}")
        return

    # --- OVERRIDE: always recompute income z from cleaned raw ---
    if var == "acs_median_income_z":
        if "acs_median_income" not in gdf_in.columns:
            print("[SKIP] missing acs_median_income (needed to recompute z)")
            return
        zcol = add_global_zscore(gdf_in, "acs_median_income")
        title = "Median Income (z, recomputed after cleaning)"
        out_name = "acs_median_income_z__recomputed_4cities.png"
    else:
        # Normal behavior:
        if var.endswith("_z"):
            zcol = var
        else:
            zcol = add_global_zscore(gdf_in, var)
        title = LABELS.get(var, var)
        out_name = f"{var}_zscore_4cities.png"

    fig = plt.figure(figsize=(20, 6))
    fig.patch.set_facecolor("white")

    gs = fig.add_gridspec(2, 4, height_ratios=[20, 1], hspace=0.05, wspace=0.05)
    axes = [fig.add_subplot(gs[0, i]) for i in range(4)]
    cax = fig.add_subplot(gs[1, :])

    for ax, city in zip(axes, CITIES):
        cg = gdf_in[gdf_in["city"] == city]

        ax.set_facecolor("white")
        cg.plot(
            ax=ax,
            column=zcol,
            cmap=CMAP,
            edgecolor="white",
            linewidth=0.4,
            vmin=Z_VMIN,
            vmax=Z_VMAX,
            alpha=1.0,
        )

        cx, cy = city_info[city]["center"]
        r = city_info[city]["radius"]
        ax.set_xlim(cx - r, cx + r)
        ax.set_ylim(cy - r, cy + r)
        ax.axis("off")
        ax.set_title(city.replace("_", " ").title(), fontsize=20, pad=10)

    norm = plt.Normalize(Z_VMIN, Z_VMAX)
    sm = plt.cm.ScalarMappable(cmap=CMAP, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, cax=cax, orientation="horizontal")
    cax.set_xticks([-2, -1, 0, 1, 2])
    cax.set_xticklabels(["-2σ", "-1σ", "0", "1σ", "2σ"], fontsize=12)
    cax.set_title("Z-score", fontsize=12, pad=6)

    fig.suptitle(title, fontsize=24, y=0.98)

    out_path = out_dir / out_name
    plt.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print("Saved:", out_path)

# ======================
# Run all
# ======================
for var in PLOT_VARS:
    plot_variable_grid(gdf, var, OUT_DIR)

print("Done. Output folder:", OUT_DIR)

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely import wkt

PROJECT_ROOT = Path(os.environ.get("PERSONALITY_SVI_ROOT", Path(__file__).resolve().parents[2]))
IN_PARQUET = f"{PROJECT_ROOT}/data/processed/model/final_dataset_zipcode_spatial2_env8_medianincome_racialdiversity.parquet"

# ---- Load ----
df = pd.read_parquet(IN_PARQUET)

# ---- Geometry fix (WKT -> shapely) ----
if "geometry" in df.columns:
    sample = df["geometry"].dropna().iloc[0]
    if isinstance(sample, str):
        df["geometry"] = df["geometry"].apply(lambda x: wkt.loads(x) if isinstance(x, str) else x)
    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")
    gdf = gdf.to_crs(epsg=3857)
else:
    gdf = None

# ---- Pick column ----
col = "acs_median_income"
if col not in df.columns:
    raise ValueError(f"Missing column: {col}")

# ---- Basic sanity filters ----
s = pd.to_numeric(df[col], errors="coerce")

print("== Basic summary ==")
print(s.describe(percentiles=[.01,.05,.1,.25,.5,.75,.9,.95,.99]))
print("\nNulls:", s.isna().sum(), " / ", len(s))

# Common suspicious patterns: <=0, tiny numbers (like 40 meaning $40k), huge (like 4000000)
print("\n== Counts in suspicious ranges ==")
print("<= 0 :", (s <= 0).sum())
print("< 1000 :", (s < 1000).sum())          # could mean it's in $k units
print("> 500000 :", (s > 500000).sum())      # very high for median HH income
print("> 2000000 :", (s > 2_000_000).sum())  # usually wrong / sentinel / join issue

# ---- Histogram (raw) ----
plt.figure(figsize=(8,4))
plt.hist(s.dropna(), bins=80)
plt.title("acs_median_income (raw) histogram")
plt.xlabel("Income value")
plt.ylabel("Count")
plt.show()

# ---- Histogram (winsorized for view) ----
clip_lo, clip_hi = np.nanpercentile(s, 1), np.nanpercentile(s, 99)
plt.figure(figsize=(8,4))
plt.hist(np.clip(s.dropna(), clip_lo, clip_hi), bins=80)
plt.title(f"acs_median_income histogram (clipped to 1–99%: {clip_lo:.0f}–{clip_hi:.0f})")
plt.xlabel("Income value (clipped)")
plt.ylabel("Count")
plt.show()

# ---- Boxplot by city (helps catch one-city join bugs) ----
if "city" in df.columns:
    tmp = df[["city", col]].copy()
    tmp[col] = pd.to_numeric(tmp[col], errors="coerce")
    # keep top N cities if it's too many
    cities = tmp["city"].value_counts().head(12).index.tolist()
    tmp = tmp[tmp["city"].isin(cities)]
    data = [tmp.loc[tmp["city"]==c, col].dropna().values for c in cities]

    plt.figure(figsize=(14,4))
    plt.boxplot(data, labels=cities, showfliers=False)
    plt.xticks(rotation=45, ha="right")
    plt.title("acs_median_income by city (boxplot, fliers hidden)")
    plt.ylabel("Income value")
    plt.show()

# ---- Check if it's actually "thousands" ----
# If median is like 40–90, it might be $k. Multiply by 1000 and re-check.
med = np.nanmedian(s)
print("\nMedian raw:", med)
if med < 500:  # heuristic
    s_k = s * 1000
    print("Heuristic: values look like 'thousands'. If *1000, median becomes:", np.nanmedian(s_k))

# ---- Compare with z (if exists) ----
zcol = "acs_median_income_z"
if zcol in df.columns:
    z = pd.to_numeric(df[zcol], errors="coerce")
    plt.figure(figsize=(6,5))
    plt.scatter(s, z, s=2, alpha=0.15)
    plt.title("acs_median_income vs acs_median_income_z")
    plt.xlabel("acs_median_income")
    plt.ylabel("acs_median_income_z")
    plt.show()

# ---- Map raw and log (if geometry exists) ----
if gdf is not None:
    plot_gdf = gdf[[col, "geometry"]].copy()
    plot_gdf[col] = pd.to_numeric(plot_gdf[col], errors="coerce")

    # Raw map (clipped 1–99% for readability)
    vmin, vmax = np.nanpercentile(plot_gdf[col], 1), np.nanpercentile(plot_gdf[col], 99)
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    plot_gdf.plot(ax=ax, column=col, legend=True, vmin=vmin, vmax=vmax, linewidth=0, edgecolor="none")
    ax.set_title(f"{col} map (clipped 1–99%: {vmin:.0f}–{vmax:.0f})")
    ax.axis("off")
    plt.show()

    # Log map (handles heavy right tail)
    plot_gdf["log10_income"] = np.log10(plot_gdf[col].where(plot_gdf[col] > 0))
    vmin2, vmax2 = np.nanpercentile(plot_gdf["log10_income"], 1), np.nanpercentile(plot_gdf["log10_income"], 99)
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    plot_gdf.plot(ax=ax, column="log10_income", legend=True, vmin=vmin2, vmax=vmax2, linewidth=0, edgecolor="none")
    ax.set_title("log10(acs_median_income) map (clipped 1–99%)")
    ax.axis("off")
    plt.show()


In [ ]:
from pathlib import Path
import matplotlib.gridspec as mgridspec

# ======================
# Combined matrix: 5 ENV features x 4 cities  (aligned with percentile grid)
# Output: reports/figures/city_maps/all_cities_features_matrix_zscore.png
# (gdf, city_info, CITIES, CMAP, Z_VMIN, Z_VMAX loaded in Cell 0)
# ======================

MATRIX_OUT = Path("<PROJECT_ROOT>/reports/figures/city_maps")
MATRIX_OUT.mkdir(parents=True, exist_ok=True)

# 5 features — matches rebuild_grid.py FEATURES_ORDER exactly
# (env_active_infra, env_active_presence, env_physical_boundaries,
#  env_symbolic_us_flag excluded to keep consistent with percentile grid)
MATRIX_FEATURES = [
    "env_building",
    "env_greenery",
    "env_open_space",
    "env_road",
    "env_vehicle_presence",
]

FEATURE_LABELS = {
    "env_building":         "Building",
    "env_greenery":         "Greenery",
    "env_open_space":       "Open Space",
    "env_road":             "Road",
    "env_vehicle_presence": "Vehicle Presence",
}

CITY_LABELS = {
    "austin_texas":       "Austin",
    "dallas_texas":       "Dallas",
    "houston_texas":      "Houston",
    "san_antonio_texas":  "San Antonio",
}

n_rows = len(MATRIX_FEATURES)
n_cols = len(CITIES)

# Precompute z-scores for all matrix features
for feat in MATRIX_FEATURES:
    zcol = f"{feat}__z"
    if zcol not in gdf.columns:
        vals = gdf[feat].to_numpy(dtype=float)
        gdf[zcol] = stats.zscore(vals, nan_policy="omit")

# Figure: n_rows map rows + 1 colorbar row
fig = plt.figure(figsize=(5 * n_cols, 4 * n_rows + 0.8))
fig.patch.set_facecolor("white")

gs = mgridspec.GridSpec(
    n_rows + 1, n_cols,
    height_ratios=[4] * n_rows + [0.25],
    hspace=0.05,
    wspace=0.05,
    figure=fig,
)

for row_idx, feat in enumerate(MATRIX_FEATURES):
    zcol = f"{feat}__z"
    for col_idx, city in enumerate(CITIES):
        ax = fig.add_subplot(gs[row_idx, col_idx])
        cg = gdf[gdf["city"] == city]

        ax.set_facecolor("white")
        cg.plot(
            ax=ax,
            column=zcol,
            cmap=CMAP,
            edgecolor="white",
            linewidth=0.3,
            vmin=Z_VMIN,
            vmax=Z_VMAX,
            alpha=1.0,
        )

        cx, cy = city_info[city]["center"]
        r = city_info[city]["radius"]
        ax.set_xlim(cx - r, cx + r)
        ax.set_ylim(cy - r, cy + r)
        ax.axis("off")

        # Column header (top row only)
        if row_idx == 0:
            ax.set_title(CITY_LABELS[city], fontsize=24, pad=10, fontweight="bold")

        # Row label (leftmost column only)
        if col_idx == 0:
            ax.text(
                -0.12, 0.5, FEATURE_LABELS[feat],
                transform=ax.transAxes,
                fontsize=24,
                va="center",
                ha="right",
                rotation=90,
            )

# Shared colorbar spanning all columns
cax = fig.add_subplot(gs[n_rows, :])
norm = plt.Normalize(Z_VMIN, Z_VMAX)
sm = plt.cm.ScalarMappable(cmap=CMAP, norm=norm)
sm.set_array([])
cb = plt.colorbar(sm, cax=cax, orientation="horizontal")
cb.set_ticks([-2, -1, 0, 1, 2])
cb.set_ticklabels(["-2\u03c3", "-1\u03c3", "0", "1\u03c3", "2\u03c3"], fontsize=12)
cax.set_title("Z-score", fontsize=12, pad=4)

out_path = MATRIX_OUT / "all_cities_features_matrix_zscore.png"
plt.savefig(out_path, dpi=200, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("Saved:", out_path)

In [ ]:
from pathlib import Path
import matplotlib.gridspec as mgridspec
from scipy import stats as _stats

# ======================
# Combined matrix: 5 Big Five traits x 4 cities
# Output: reports/figures/city_maps/all_cities_traits_matrix_zscore.png
# (gdf, city_info, CITIES, CMAP, Z_VMIN, Z_VMAX loaded in Cell 0)
# ======================

MATRIX_OUT = Path("<PROJECT_ROOT>/reports/figures/city_maps")
MATRIX_OUT.mkdir(parents=True, exist_ok=True)

MATRIX_TRAITS = [
    "openness_mean",
    "conscientiousness_mean",
    "extraversion_mean",
    "agreeableness_mean",
    "neuroticism_mean",
]

TRAIT_LABELS = {
    "openness_mean":         "Openness",
    "conscientiousness_mean": "Conscientiousness",
    "extraversion_mean":     "Extraversion",
    "agreeableness_mean":    "Agreeableness",
    "neuroticism_mean":      "Neuroticism",
}

CITY_LABELS_T = {
    "austin_texas":       "Austin",
    "dallas_texas":       "Dallas",
    "houston_texas":      "Houston",
    "san_antonio_texas":  "San Antonio",
}

n_rows = len(MATRIX_TRAITS)
n_cols = len(CITIES)

# Precompute global z-scores for all traits
for trait in MATRIX_TRAITS:
    zcol = f"{trait}__z"
    if zcol not in gdf.columns:
        vals = gdf[trait].to_numpy(dtype=float)
        gdf[zcol] = _stats.zscore(vals, nan_policy="omit")

# Figure: n_rows map rows + 1 colorbar row
fig = plt.figure(figsize=(5 * n_cols, 4 * n_rows + 0.8))
fig.patch.set_facecolor("white")

gs = mgridspec.GridSpec(
    n_rows + 1, n_cols,
    height_ratios=[4] * n_rows + [0.25],
    hspace=0.05,
    wspace=0.05,
    figure=fig,
)

for row_idx, trait in enumerate(MATRIX_TRAITS):
    zcol = f"{trait}__z"
    for col_idx, city in enumerate(CITIES):
        ax = fig.add_subplot(gs[row_idx, col_idx])
        cg = gdf[gdf["city"] == city]

        ax.set_facecolor("white")
        cg.plot(
            ax=ax,
            column=zcol,
            cmap=CMAP,
            edgecolor="white",
            linewidth=0.3,
            vmin=Z_VMIN,
            vmax=Z_VMAX,
            alpha=1.0,
        )

        cx, cy = city_info[city]["center"]
        r = city_info[city]["radius"]
        ax.set_xlim(cx - r, cx + r)
        ax.set_ylim(cy - r, cy + r)
        ax.axis("off")

        # Column header (top row only)
        if row_idx == 0:
            ax.set_title(CITY_LABELS_T[city], fontsize=24, pad=10, fontweight="bold")

        # Row label (leftmost column only)
        if col_idx == 0:
            ax.text(
                -0.12, 0.5, TRAIT_LABELS[trait],
                transform=ax.transAxes,
                fontsize=24,
                va="center",
                ha="right",
                rotation=90,
            )

# Shared colorbar
cax = fig.add_subplot(gs[n_rows, :])
norm = plt.Normalize(Z_VMIN, Z_VMAX)
sm = plt.cm.ScalarMappable(cmap=CMAP, norm=norm)
sm.set_array([])
cb = plt.colorbar(sm, cax=cax, orientation="horizontal")
cb.set_ticks([-2, -1.5, -1, -0.5, 0, 0.5, 1, 1.5, 2])
cb.set_ticklabels(["-2\u03c3", "-1.5\u03c3", "-1\u03c3", "-0.5\u03c3", "0", "0.5\u03c3", "1\u03c3", "1.5\u03c3", "2\u03c3"], fontsize=10)
cax.set_title("Z-score", fontsize=12, pad=4)

out_path = MATRIX_OUT / "all_cities_traits_matrix_zscore.png"
plt.savefig(out_path, dpi=200, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("Saved:", out_path)


In [ ]:
import matplotlib.patches as mpatches

# ======================
# Appendix: ZIP code model inclusion/exclusion map
# Output: reports/figures/appendix_maps/zipcode_model_inclusion_mask.png
# (gdf loaded in Cell 0 — all ZIP codes, no model filter applied)
# ======================

APPENDIX_OUT = Path("<PROJECT_ROOT>/reports/figures/appendix_maps")
APPENDIX_OUT.mkdir(parents=True, exist_ok=True)

# V2 model filter thresholds (must match OLS_SpatialLag_Report_Optimized_v2_post2010.ipynb)
MIN_PARTICIPANT   = 20
MIN_GRID_COVERAGE = 0.03

mask_pc   = gdf["participant_count"]  >= MIN_PARTICIPANT
mask_grid = gdf["grid_coverage_mean"] >= MIN_GRID_COVERAGE

def _status(pc_ok, grid_ok):
    if pc_ok and grid_ok:      return "included"
    if not pc_ok and grid_ok:  return "excl_count"
    if pc_ok and not grid_ok:  return "excl_grid"
    return "excl_both"

gdf["_status"] = [_status(pc, gr) for pc, gr in zip(mask_pc, mask_grid)]

n = {s: int((gdf["_status"] == s).sum())
     for s in ["included", "excl_count", "excl_grid", "excl_both"]}
print("Inclusion counts:", n)

STATUS_COLORS = {
    "included":   "#2ca02c",
    "excl_count": "#d62728",
    "excl_grid":  "#ff7f0e",
    "excl_both":  "#9467bd",
}
STATUS_LABELS = {
    "included":   f"Included in model  (n={n['included']})",
    "excl_count": f"Excluded: participant count < {MIN_PARTICIPANT}  (n={n['excl_count']})",
    "excl_grid":  f"Excluded: grid coverage < {MIN_GRID_COVERAGE}  (n={n['excl_grid']})",
    "excl_both":  f"Excluded: both filters  (n={n['excl_both']})",
}
DRAW_ORDER = ["included", "excl_count", "excl_grid", "excl_both"]

fig, axes = plt.subplots(1, len(CITIES), figsize=(5 * len(CITIES), 5.5))
fig.patch.set_facecolor("white")

for col_idx, city in enumerate(CITIES):
    ax = axes[col_idx]
    cg = gdf[gdf["city"] == city]
    ax.set_facecolor("white")
    for status in DRAW_ORDER:
        sub = cg[cg["_status"] == status]
        if len(sub):
            sub.plot(ax=ax, color=STATUS_COLORS[status],
                     edgecolor="white", linewidth=0.3, alpha=0.9)
    cx, cy = city_info[city]["center"]
    r = city_info[city]["radius"]
    ax.set_xlim(cx - r, cx + r)
    ax.set_ylim(cy - r, cy + r)
    ax.axis("off")
    ax.set_title(CITY_LABELS[city], fontsize=14, pad=6, fontweight="bold")

patches = [mpatches.Patch(color=STATUS_COLORS[s], label=STATUS_LABELS[s])
           for s in DRAW_ORDER]
fig.legend(handles=patches, loc="lower center", ncol=2,
           fontsize=10, framealpha=0.92, bbox_to_anchor=(0.5, -0.10))

n_total = sum(n.values())
fig.suptitle(
    f"ZIP code model inclusion \u2014 V2 (post-2010)  "
    f"[total: {n_total}  |  included: {n['included']}  |  excluded: {n_total - n['included']}]",
    fontsize=13, fontweight="bold", y=1.02,
)

out_path = APPENDIX_OUT / "zipcode_model_inclusion_mask.png"
plt.savefig(out_path, dpi=200, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("Saved:", out_path)